In [ ]:
import math
import pandas as pd
import folium
from folium.plugins import PolyLineTextPath
from folium.features import DivIcon
from google.cloud import bigquery
from google.cloud import storage

# --- TACTICAL BLOCKLIST ---
# Add ship names or MMSIs here to permanently drop them from the render
KNOWN_SPOOFERS = [
    'CSTAR VOYAGER'
]

# --- Helper Function ---
def haversine_nm(lat1, lon1, lat2, lon2):
    R = 3440.065
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# 1. Initialize GCP Clients
project_id = 'your-gcp-project-id'  # <-- REPLACE with your GCP project ID
bq_client = bigquery.Client(project=project_id)
storage_client = storage.Client(project=project_id)

# 2. Define Queries
query_current = """
WITH LatestVesselPings AS (
  SELECT mmsi, ship_name, latitude, longitude, speed_over_ground, timestamp,
    ROW_NUMBER() OVER(PARTITION BY mmsi ORDER BY timestamp DESC) as ping_rank
  FROM `hormuz_conflict_data.live_vessel_tracking`
  WHERE latitude BETWEEN 23.0 AND 28.0 AND longitude BETWEEN 53.0 AND 58.0
)
SELECT
  mmsi,
  COALESCE(NULLIF(ship_name, ''), 'UNKNOWN_VESSEL') AS vessel_name,
  latitude, longitude, speed_over_ground AS speed_knots,
  timestamp AS last_seen_utc,
  CASE
    WHEN speed_over_ground < 0.5 THEN 'Ghost Fleet'
    WHEN speed_over_ground BETWEEN 0.5 AND 3.5 THEN 'Caution'
    ELSE 'Active Transit'
  END AS tactical_status
FROM LatestVesselPings
WHERE ping_rank = 1
"""

query_snapshot = """
WITH PastVesselPings AS (
  SELECT mmsi, latitude, longitude, timestamp,
    ROW_NUMBER() OVER(PARTITION BY mmsi ORDER BY timestamp DESC) as ping_rank
  FROM `hormuz_conflict_data.live_vessel_tracking-2026-03-17T08_40_05`
  WHERE latitude BETWEEN 23.0 AND 28.0 AND longitude BETWEEN 53.0 AND 58.0
)
SELECT mmsi, latitude, longitude, timestamp AS last_seen_utc
FROM PastVesselPings
WHERE ping_rank = 1
"""

# 3. Fetch Data
print("Fetching current data...")
df_current = bq_client.query(query_current).to_dataframe()

print("Fetching snapshot data...")
df_snapshot = bq_client.query(query_snapshot).to_dataframe()

latest_ping_time = df_current['last_seen_utc'].max().strftime('%Y-%m-%d %H:%M:%S UTC')

# 4. Merge the DataFrames
df_merged = pd.merge(
    df_snapshot,
    df_current,
    on='mmsi',
    suffixes=('_past', '_curr')
)

# 5. Build the Interactive Map
map_center = [25.390093, 55.270566]
vessel_map = folium.Map(location=map_center, zoom_start=10, tiles=None)

# --- Add Basemap Layers ---
folium.TileLayer('CartoDB Dark_Matter', name='Tactical (Dark)').add_to(vessel_map)
folium.TileLayer('OpenStreetMap', name='Standard Map').add_to(vessel_map)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Satellite (Esri)',
    overlay=False
).add_to(vessel_map)

# 6. Draw the Map Elements
drawn_vectors = 0
dropped_anomalies = 0
dropped_blocklist = 0

for index, row in df_merged.iterrows():

    # 1. EXPLICIT BLOCKLIST CHECK
    if row['vessel_name'] in KNOWN_SPOOFERS:
        dropped_blocklist += 1
        continue

    past_coords = [row['latitude_past'], row['longitude_past']]
    curr_coords = [row['latitude_curr'], row['longitude_curr']]

    distance_nm = haversine_nm(past_coords[0], past_coords[1], curr_coords[0], curr_coords[1])

    # 2. Jitter Filter (Drop stationary wobble < 0.05 NM)
    if distance_nm < 0.05:
        continue

    # 3. Safe Physics Filter (Implied speed > 45 knots)
    time_diff_hours = (row['last_seen_utc_curr'] - row['last_seen_utc_past']).total_seconds() / 3600.0
    implied_speed_knots = 0
    if time_diff_hours > 0:
        implied_speed_knots = distance_nm / time_diff_hours

    if implied_speed_knots > 45:
        dropped_anomalies += 1
        continue
    # -----------------------------------

    drawn_vectors += 1

    color_map = {
        'Ghost Fleet': '#ff4d4d',
        'Caution': '#ffcc00',
        'Active Transit': '#33cc33'
    }
    vec_color = color_map.get(row['tactical_status'], '#ffffff')

    # Original Position
    folium.CircleMarker(location=past_coords, radius=2, color='#888888', fill=True, fill_opacity=0.5).add_to(vessel_map)

    # Vector Line
    vector_line = folium.PolyLine(
        locations=[past_coords, curr_coords], weight=2, color=vec_color, opacity=0.8,
        tooltip=f"{row['vessel_name']}: Moved {distance_nm:.2f} NM"
    )
    vector_line.add_to(vessel_map)

    PolyLineTextPath(
        vector_line, '  ►', repeat=False, offset=5,
        attributes={'fill': vec_color, 'font-weight': 'bold', 'font-size': '16'}
    ).add_to(vessel_map)

    # Current Position
    folium.CircleMarker(
        location=curr_coords, radius=4, color=vec_color, fill=True, fill_opacity=1.0,
        tooltip=f"MMSI: {row['mmsi']} | Speed: {row['speed_knots']} knots"
    ).add_to(vessel_map)

    # Popup Dossier
    popup_html = f"""
    <div style='font-family: Arial, sans-serif; min-width: 180px;'>
        <h4 style='margin-top: 0; margin-bottom: 5px; color: {vec_color};'>{row['vessel_name']}</h4>
        <hr style='margin: 5px 0;'>
        <b>MMSI:</b> {row['mmsi']}<br>
        <b>Status:</b> {row['tactical_status']}<br>
        <b>Reported Speed:</b> {row['speed_knots']} knots<br>
        <b>Avg Transit Speed:</b> {implied_speed_knots:.1f} knots<br>
        <b>Moved:</b> {distance_nm:.2f} NM<br>
        <b>Last Ping:</b> {row['last_seen_utc_curr'].strftime('%H:%M:%S UTC')}
    </div>
    """

    folium.Marker(
        location=curr_coords,
        icon=DivIcon(
            icon_size=(150,36), icon_anchor=(-8, 10),
            html=f'<div style="font-size: 8pt; color: white; text-shadow: 1px 1px 2px black; white-space: nowrap; cursor: pointer;">{row["vessel_name"]}</div>',
        ),
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(vessel_map)

# --- Add Layer Switcher ---
folium.LayerControl(position='topright').add_to(vessel_map)

# --- Inject Floating Refresh Timestamp Dashboard ---
refresh_html = f"""
<div style="position: fixed; bottom: 30px; left: 30px; width: 230px;
            background-color: rgba(30, 30, 30, 0.8); border: 1px solid #555;
            z-index: 9999; font-family: Arial, sans-serif; font-size: 12px;
            color: #eeeeee; border-radius: 5px; padding: 10px; box-shadow: 2px 2px 5px rgba(0,0,0,0.5);">
    <strong style="color: #4CAF50;">● LIVE TACTICAL FEED</strong><br>
    <hr style="border: 0; border-top: 1px solid #555; margin: 5px 0;">
    <b>Latest Ping:</b> {latest_ping_time}<br><br>
    <b>Active Vectors:</b> {drawn_vectors}<br>
    <span style="color: #ffcc00;"><b>Physics Drops:</b> {dropped_anomalies}</span><br>
    <span style="color: #ff4d4d;"><b>Blocklist Drops:</b> {dropped_blocklist}</span>
</div>
"""
vessel_map.get_root().html.add_child(folium.Element(refresh_html))

print(f"Mapped {drawn_vectors} clean vectors. Physics dropped {dropped_anomalies}, Blocklist dropped {dropped_blocklist}.")

import time

# 7. Export HTML directly to GCS Bucket
bucket_name = "your-gcs-bucket-name"  # <-- REPLACE with your GCS bucket name
file_path = "ships/tactical_map.html"

map_html_string = vessel_map.get_root().render()

bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(file_path)

# 1. Tell GCS not to cache it (Server-side)
blob.cache_control = 'no-store, no-cache, must-revalidate, max-age=0'
blob.upload_from_string(map_html_string, content_type='text/html')

# 2. THE CACHE-BUSTER: Generate a unique UNIX timestamp
unique_timestamp = int(time.time())

# 3. Build the URL with the dummy '?v=' parameter (Client-side bypass)
# Note: If your bucket isn't strictly public, you might need a signed URL instead.
bypassed_url = f"https://storage.googleapis.com/{bucket_name}/{file_path}?v={unique_timestamp}"

print(f"✅ Map successfully exported to GCS.")
print(f"🔗 CLICK HERE for the live, un-cached map:\n{bypassed_url}")

# 8. Display Map inline
vessel_map
# 8. Display Map inline
vessel_map

Fetching current data...
Fetching snapshot data...
Mapped 35 clean vectors. Physics dropped 0, Blocklist dropped 1.
✅ Map successfully exported to GCS.
🔗 CLICK HERE for the live, un-cached map:
https://storage.googleapis.com/hormuz-demo-2026/ships/tactical_map.html?v=1773739594
